# 🌍 Hackathon Submission: Multi-City Air Quality Monitoring and Spatial Analysis Platform

### Project Overview
This notebook demonstrates a modern, end-to-end spatial analytics pipeline that connects public serverless data lakes with advanced interactive visualizations, localized grid simulations, and 3D surface interpolation modeling.

### What We Built:
1. **Direct AWS S3 Data Lakehouse Connection**: An in-memory, serverless connection querying public CSV archives directly using DuckDB without regional limits.
2. **Multi-Tier Mesh Grid Simulation**: A synthetic spatial coordinate generator reflecting actual physical density classes (Dense Urban Centers to Monitoring Deserts).
3. **Interactive Local Map & Diagnostics**: A combined widget dashboard correlating raw values on spatial maps with local statistical histograms.
4. **Continuous 3D Atmospheric Surface Modeler**: A spatial interpolation engine implementing the Haversine trigonometric formula and SciPy cubic gridding to turn discrete points into continuous 3D pollutant concentrations.
5. **Secure Premium Maps Authentication**: Integrated Google Colab Secrets to safely load Mapbox API tokens, elevating cartographic base maps safely without hardcoding.
6. **Global Real-Time Search Dashboard**: A fully dynamic panel allowing users to search any global address, query live measurements from OpenAQ, compute spatial resolution metrics, and map them with premium vector layers.

### Step 1: Serverless S3 Database Initialization

* **What the Code is Doing**: This block installs and loads DuckDB along with its high-performance S3 filesystem and HTTP transport modules. It initializes an in-memory database instance ready to query remote open data archives.
* **Services Used**: DuckDB, AWS Open Data Registry (OpenAQ S3 bucket).
* **Methods & Steps**: Initializes an in-memory DuckDB connection, installs the native `httpfs` extension, and configures the default S3 region to match the hosting region of the OpenAQ bucket.
* **Assumptions & Approximations**: Assumes the current server has active internet access to resolve and fetch native compiled binaries of DuckDB extensions, and that the external open S3 bucket is in its public-read configuration.

In [ ]:
!pip install duckdb -q
import duckdb
import json
import math

con = duckdb.connect()
# Enable anonymous HTTP/S3 reads from the Registry of Open Data on AWS
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-east-1';")
print("DuckDB ready with S3 open data connector!")

### Step 2: Serverless Historical Data Extraction

* **What the Code is Doing**: Executes a direct, partition-pruned SQL query on highly-compressed remote `.csv.gz` logs stored inside the public OpenAQ S3 bucket, outputting historical readings in a tabular format.
* **Services Used**: DuckDB, AWS S3 serverless analytics interface.
* **Methods & Steps**: Utilizes DuckDB's `read_csv_auto` wrapper to read a specific geographic location directory structure inside the remote partitioned structure directly into memory and extracts a Pandas DataFrame.
* **Assumptions & Approximations**: Assumes the target file path exists and that historical structures are grouped by standard hive location keys. No geographical coordinate projections or physical conversions are performed at this step.

In [ ]:
# Query live OpenAQ Hive-partitioned records directly from AWS without credentials
sample_query = """
SELECT
    location,
    parameter,
    value,
    units,
    datetime,
    lat,
    lon
FROM read_csv_auto('s3://openaq-data-archive/records/csv.gz/locationid=2178/year=2023/month=02/*.csv.gz')
LIMIT 5;
"""
df_sample = con.execute(sample_query).df()
display(df_sample)

### Step 3: Spatial Benchmark Grid Generation

* **What the Code is Doing**: Constructs an active benchmark grid spanning diverse geographical categories across multiple cities worldwide to model varying densities of monitoring networks.
* **Services Used**: Native Python standard libraries (`math`, `json`).
* **Methods & Steps**: Loops through regional boundary boxes with step sizes based on network densities (e.g., small step sizes for dense urban areas, wide step size for rural deserts). It uses spatial coordinate distance formulas to model smooth regional pollutant declines combined with mathematical noise functions.
* **Assumptions & Approximations**: Uses standard 2D flat Cartesian distance calculations to model local air pollution decay rates relative to coordinate centers. Earth curvature offsets are considered negligible for the tight bounds used.

In [ ]:
# Generate the multi-city benchmark grid with both Dense Hubs and Sensor Deserts
CITIES = {
    "Delhi NCR": {"center": [77.2090, 28.6139], "bbox": [76.90, 28.40, 77.45, 28.85], "base": 88.0, "var": 35.0, "tier": "Dense Monitoring"},
    "London": {"center": [-0.1276, 51.5074], "bbox": [-0.35, 51.35, 0.15, 51.65], "base": 14.5, "var": 6.0, "tier": "Dense Monitoring"},
    "Los Angeles": {"center": [-118.2437, 34.0522], "bbox": [-118.55, 33.80, -118.05, 34.25], "base": 22.0, "var": 12.0, "tier": "Dense Monitoring"},
    "Hooghly Industrial Corridor": {"center": [88.3639, 22.5726], "bbox": [88.15, 22.40, 88.50, 22.85], "base": 64.0, "var": 24.0, "tier": "Moderate Monitoring"},
    "Rural Fringe (Sensor Desert)": {"center": [87.5000, 23.2000], "bbox": [87.20, 23.00, 87.80, 23.40], "base": 72.0, "var": 40.0, "tier": "Sensor Desert"}
}

features = []
station_idx = 1

for city, cfg in CITIES.items():
    step = 0.04 if cfg["tier"] != "Sensor Desert" else 0.14
    min_lon, min_lat, max_lon, max_lat = cfg["bbox"]
    lat = min_lat
    while lat <= max_lat:
        lon = min_lon
        while lon <= max_lon:
            dist = math.sqrt((lon - cfg["center"][0])**2 + (lat - cfg["center"][1])**2)
            noise = math.sin(lon * 45) * math.cos(lat * 45)
            pm25 = max(5.0, round(cfg["base"] + (cfg["var"] * noise) + (dist * 12), 1))

            risk = "HAZARDOUS" if pm25 >= 60 else ("UNHEALTHY" if pm25 >= 35 else ("MODERATE" if pm25 >= 15 else "GOOD"))

            features.append({
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [round(lon, 4), round(lat, 4)]},
                "properties": {
                    "id": f"station_{station_idx}",
                    "city": city,
                    "pm25": pm25,
                    "coverage_tier": cfg["tier"],
                    "risk_tier": risk,
                    "parameter": "pm25",
                    "unit": "µg/m³"
                }
            })
            station_idx += 1
            lon += step
        lat += step

geojson = {"type": "FeatureCollection", "features": features}

with open("sensor_grid.geojson", "w") as f:
    json.dump(geojson, f, indent=2)

print(f"Generated sensor_grid.geojson with {len(features)} stations!")

### Step 4: Interactive Local Diagnostic Dashboard

* **What the Code is Doing**: Combines interactive geographic rendering using Folium alongside diagnostic histogram charts using Matplotlib and Seaborn, and connects them with standard IPyWidgets controls.
* **Services Used**: Folium, CartoDB Positron maps, Matplotlib, Seaborn, IPyWidgets.
* **Methods & Steps**: Sets up a dropdown controller linked to rendering handlers. When a user selects a city, the map center updates, markers are color-coded by EPA risk categories, and the matching PM2.5 histogram is recalculated.
* **Assumptions & Approximations**: Assumes that CartoDB's free, anonymous Positron tileset is reachable via standard HTTPS protocols and that the local system is capable of rendering JavaScript widgets natively.

In [ ]:
import json
import os
import math
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. Generate sensor_grid.geojson if it does not exist ---
geojson_filename = 'sensor_grid.geojson'
if not os.path.exists(geojson_filename):
    CITIES = {
        "Delhi NCR": {"center": [77.2090, 28.6139], "bbox": [76.90, 28.40, 77.45, 28.85], "base": 88.0, "var": 35.0, "tier": "Dense Monitoring"},
        "London": {"center": [-0.1276, 51.5074], "bbox": [-0.35, 51.35, 0.15, 51.65], "base": 14.5, "var": 6.0, "tier": "Dense Monitoring"},
        "Los Angeles": {"center": [-118.2437, 34.0522], "bbox": [-118.55, 33.80, -118.05, 34.25], "base": 22.0, "var": 12.0, "tier": "Dense Monitoring"},
        "Hooghly Industrial Corridor": {"center": [88.3639, 22.5726], "bbox": [88.15, 22.40, 88.50, 22.85], "base": 64.0, "var": 24.0, "tier": "Moderate Monitoring"},
        "Rural Fringe (Sensor Desert)": {"center": [87.5000, 23.2000], "bbox": [87.20, 23.00, 87.80, 23.40], "base": 72.0, "var": 40.0, "tier": "Sensor Desert"}
    }

    features = []
    station_idx = 1

    for city, cfg in CITIES.items():
        step = 0.04 if cfg["tier"] != "Sensor Desert" else 0.14
        min_lon, min_lat, max_lon, max_lat = cfg["bbox"]
        lat = min_lat
        while lat <= max_lat:
            lon = min_lon
            while lon <= max_lon:
                # Euclidean distance calculation relative to the city center
                dist = math.sqrt((lon - cfg["center"][0])**2 + (lat - cfg["center"][1])**2)
                noise = math.sin(lon * 45) * math.cos(lat * 45)
                pm25 = max(5.0, round(cfg["base"] + (cfg["var"] * noise) + (dist * 12), 1))
                risk = "HAZARDOUS" if pm25 >= 60 else ("UNHEALTHY" if pm25 >= 35 else ("MODERATE" if pm25 >= 15 else "GOOD"))

                features.append({
                    "type": "Feature",
                    "geometry": {"type": "Point", "coordinates": [round(lon, 4), round(lat, 4)]},
                    "properties": {
                        "id": f"station_{station_idx}",
                        "city": city,
                        "pm25": pm25,
                        "coverage_tier": cfg["tier"],
                        "risk_tier": risk,
                        "parameter": "pm25",
                        "unit": "µg/m³"
                    }
                })
                station_idx += 1
                lon += step
            lat += step

    geojson = {"type": "FeatureCollection", "features": features}
    with open(geojson_filename, "w") as f:
        json.dump(geojson, f, indent=2)

# --- 2. Load the GeoJSON into a Pandas DataFrame ---
with open(geojson_filename, 'r') as f:
    geojson_data = json.load(f)

rows = []
for feature in geojson_data['features']:
    props = feature['properties']
    coords = feature['geometry']['coordinates']
    rows.append({
        'id': props['id'],
        'city': props['city'],
        'pm25': props['pm25'],
        'coverage_tier': props['coverage_tier'],
        'risk_tier': props['risk_tier'],
        'lon': coords[0],
        'lat': coords[1]
    })

df_sensors = pd.DataFrame(rows)

# 3. Define risk colors mapping
COLOR_MAP = {
    'GOOD': 'green',
    'MODERATE': 'orange',
    'UNHEALTHY': 'red',
    'HAZARDOUS': 'purple'
}

# 4. Create interactive widgets
city_dropdown = widgets.Dropdown(
    options=sorted(df_sensors['city'].unique()),
    value='Delhi NCR',
    description='Select City:',
    style={'description_width': 'initial'}
)

output_map = widgets.Output()
output_plot = widgets.Output()

def update_dashboard(change):
    selected_city = change['new'] if change else city_dropdown.value
    city_df = df_sensors[df_sensors['city'] == selected_city]

    # ---- Draw Plot ----
    with output_plot:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.histplot(data=city_df, x='pm25', hue='risk_tier', palette=COLOR_MAP,
                     multiple='stack', bins=15, ax=ax, kde=True)
        ax.set_title(f'PM2.5 Distribution in {selected_city} (Coverage: {city_df["coverage_tier"].iloc[0]})')
        ax.set_xlabel('PM2.5 Concentration (µg/m³)')
        ax.set_ylabel('Sensor Count')
        plt.tight_layout()
        plt.show()

    # ---- Draw Map ----
    with output_map:
        clear_output(wait=True)
        center_lat = city_df['lat'].mean()
        center_lon = city_df['lon'].mean()

        m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='CartoDB positron')

        for _, row in city_df.iterrows():
            popup_html = f"""
            <h4>Station: {row['id']}</h4>
            <b>City:</b> {row['city']}<br>
            <b>PM2.5:</b> {row['pm25']} µg/m³<br>
            <b>Risk Level:</b> <span style='color:{COLOR_MAP[row['risk_tier']]} '>{row['risk_tier']}</span><br>
            <b>Type:</b> {row['coverage_tier']}
            """
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=8,
                popup=folium.Popup(popup_html, max_width=300),
                color=COLOR_MAP[row['risk_tier']],
                fill=True,
                fill_color=COLOR_MAP[row['risk_tier']],
                fill_opacity=0.7
            ).add_to(m)
        display(m)

city_dropdown.observe(update_dashboard, names='value')

display(city_dropdown)
display(widgets.VBox([output_plot, output_map]))
update_dashboard(None)

### Step 5: Continuous 3D Atmospheric Surface Modeling

* **What the Code is Doing**: Generates an advanced continuous 3D surface model representing local atmospheric concentration maps, transforming discrete station readings into continuous spatial grids.
* **Services Used**: NumPy, SciPy (Spatial Interpolation module), Matplotlib 3D toolkit.
* **Methods & Steps**: Computes distances between the target location center and stations using the Haversine formula, filters stations, interpolates missing coordinates on a 100x100 grid using cubic interpolation, and visualizes the result as a colored 3D surface.
* **Assumptions & Approximations**: Employs the mathematical **Haversine formula** to approximate true geographic distances across the spherical surface of the earth. Fills out-of-bounds boundary values outside of the sensor convex hull using the localized mean value.

### Dynamic 3D Air Quality Surface Modeling

This interface allows you to:
1. Select a focus area and define a search radius (in kilometers, approximated from degrees).
2. Render a 3D spatial plot where the X and Y axes represent the physical layout of the region, and the Z-axis represents the PM2.5/AQI level.
3. Interpolate the sensor data to create a smooth, curved surface showing how air quality moves and concentrates across the region.

In [ ]:
import numpy as np
from scipy.interpolate import griddata
from mpl_toolkits.mplot3d import Axes3D

# Create interactive widgets
focus_city_dropdown = widgets.Dropdown(
    options=sorted(df_sensors['city'].unique()),
    value='Delhi NCR',
    description='Center City:',
    style={'description_width': 'initial'}
)

radius_slider = widgets.FloatSlider(
    value=25.0,
    min=5.0,
    max=50.0,
    step=2.5,
    description='Radius (km):',
    style={'description_width': 'initial'}
)

output_3d = widgets.Output()

def haversine_distance(lat1, lon1, lat2, lon2):
    # Approximates distance in km
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

def update_3d_surface(change):
    selected_city = focus_city_dropdown.value
    radius_km = radius_slider.value

    # Get coordinates of center location
    city_center = df_sensors[df_sensors['city'] == selected_city]
    center_lat = city_center['lat'].mean()
    center_lon = city_center['lon'].mean()

    # Calculate distances from center
    df_sensors['dist_from_center'] = haversine_distance(center_lat, center_lon, df_sensors['lat'], df_sensors['lon'])
    filtered_df = df_sensors[df_sensors['dist_from_center'] <= radius_km]

    with output_3d:
        clear_output(wait=True)

        if len(filtered_df) < 4:
            print("Not enough sensors within this radius to interpolate a 3D surface. Please expand the radius.")
            return

        # Prepare coordinates
        x = filtered_df['lon'].values
        y = filtered_df['lat'].values
        z = filtered_df['pm25'].values

        # Create a grid for the 3D surface
        grid_x, grid_y = np.mgrid[
            x.min():x.max():100j,
            y.min():y.max():100j
        ]

        # Interpolate the AQI (PM2.5) levels on the grid to construct a curved plane
        grid_z = griddata((x, y), z, (grid_x, grid_y), method='cubic')

        # Handle edge NaNs if interpolation goes outside convex hull
        grid_z = np.nan_to_num(grid_z, nan=np.nanmean(z))

        # Plotting
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subject(111, projection='3d') if hasattr(fig, 'add_subject') else fig.add_subplot(111, projection='3d')

        # Plot the smooth interpolated surface
        surf = ax.plot_surface(grid_x, grid_y, grid_z, cmap='YlOrRd', edgecolor='none', alpha=0.85)

        # Plot actual sensor points as scatter on top
        ax.scatter(x, y, z, color='black', s=25, label='Reporting Sensors', depthshade=True)

        ax.set_title(f'3D AQI (PM2.5) Surface Dispersion Profile\n({selected_city} - {radius_km}km radius)', fontsize=14, pad=15)
        ax.set_xlabel('Longitude (°E)', labelpad=10)
        ax.set_ylabel('Latitude (°N)', labelpad=10)
        ax.set_zlabel('PM2.5 Level (µg/m³)', labelpad=10)

        # Add color bar to illustrate risk density
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='PM2.5 Concentration')
        ax.legend()

        # Adjust viewing angle for perspective depth
        ax.view_init(elev=30, azim=45)

        plt.tight_layout()
        plt.show()

# Observe and link
focus_city_dropdown.observe(update_3d_surface, names='value')
radius_slider.observe(update_3d_surface, names='value')

# Display Interface
controls = widgets.HBox([focus_city_dropdown, radius_slider])
display(controls)
display(output_3d)
update_3d_surface(None)

### Step 6: Secure Mapbox API Authentication

* **What the Code is Doing**: Connects securely to the Google Colab credential storage manager, fetching private API tokens required to utilize premium Mapbox cartography styles.
* **Services Used**: Google Colab Userdata Secrets Manager, Mapbox Style APIs.
* **Methods & Steps**: Dynamically attempts to load private environmental variables (`MAP_API_KEY`) using `userdata.get()`. If keys are missing, it gracefully configures local parameters to default to free, public fallbacks.
* **Assumptions & Approximations**: Assumes the user has configured the `MAP_API_KEY` parameter inside their Google Colab key manager (symbolized by the Key icon on the left-side toolbar).

### Securely Loading Spatial & Cloud Credentials
We use the official `google.colab.userdata` module to securely load our map API keys or AWS credentials into the runtime environment without printing or exposing them in the notebook source.

In [ ]:
import os
try:
    from google.colab import userdata

    # Secure loading of Mapbox premium styles base token via Google Secrets
    MAP_API_KEY = userdata.get('MAP_API_KEY')

    if MAP_API_KEY:
        # Modified print statement to ensure no sensitive key context is printed
        print("✅ Mapbox API key successfully detected and securely loaded in memory. (Value hidden for security)")
    else:
        print("ℹ️ Note: MAP_API_KEY environment secret is empty. Falling back to public anonymous map tile limits.")
except Exception as e:
    print("ℹ️ Note: Colab Secrets context not found. Defaulting map components to public anonymous limits.")

### Step 7: End-to-End Live Global Search & Sensor Density Dashboard

* **What the Code is Doing**: This dashboard provides real-time search functionality. It takes any global textual input (like 'Paris' or 'Tokyo'), geocodes it, searches for real-time sensor stations within a chosen radius from the OpenAQ API, and maps them. If live stations are not available, it dynamically simulates a localized environmental monitoring network on the fly.
* **Services Used**: OpenAQ REST API, Nominatim Geocoding API, Mapbox Maps API (via private token fallback), Folium Leaflet maps.
* **Methods & Steps**: Sends safe HTTP requests to OpenStreetMap's geocoding engine, fetches real-time environmental measurements, calculates local spatial resolution (the minimum pairwise physical distance between stations), and maps them on high-contrast Mapbox dark tiles or standard CartoDB layers.
* **Assumptions & Approximations**: Employs standard Great-Circle trigonometric formulas to calculate precise minimum physical distances (spatial resolution) between global stations in kilometers.

### Dynamic Global Air Quality & Sensor Density Mapping

This dashboard connects directly to the **OpenAQ API** to fetch real-time global PM2.5 observations based on any custom address or city input.

It dynamically calculates:
- **Sensor Density**: The number of sensors per square kilometer in your search radius.
- **Sensor Resolution**: The minimum distance between any two reporting station pairs in the area (revealing monitoring gaps).
- **AQI Overlay**: A semi-transparent risk layer colored according to current concentration levels, positioned directly on top of the actual geographic base map.

In [ ]:
import urllib.request
import json
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import userdata

# --- Input Widgets ---
address_input = widgets.Text(
    value='London',
    placeholder='Enter any city, address, or lat/lon',
    description='Location:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

radius_km_input = widgets.FloatSlider(
    value=25.0,
    min=5.0,
    max=100.0,
    step=5.0,
    description='Radius (km):',
    style={'description_width': 'initial'}
)

search_button = widgets.Button(
    description='Fetch Live Air Quality',
    button_style='primary',
    icon='search'
)

metrics_output = widgets.Output()
dashboard_output = widgets.Output()

def geocode_address(query):
    """Geocode address using Nominatim API."""
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(query)}&format=json&limit=1"
    req = urllib.request.Request(url, headers={'User-Agent': 'ColabAQIDashboard/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data:
                return float(data[0]['lat']), float(data[0]['lon']), data[0]['display_name']
    except Exception as e:
        pass
    return None

def fetch_openaq_data(lat, lon, radius_km):
    """Fetch live PM2.5 measurements within radius from OpenAQ API."""
    radius_meters = int(radius_km * 1000)
    url = f"https://api.openaq.org/v2/measurements?coordinates={lat},{lon}&radius={radius_meters}&parameter=pm25&limit=100"
    req = urllib.request.Request(url, headers={'User-Agent': 'ColabAQIDashboard/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            res_data = json.loads(response.read().decode())
            results = res_data.get('results', [])
            records = []
            for r in results:
                coords = r.get('coordinates', {})
                if coords:
                    records.append({
                        'location': r.get('location', 'Unknown'),
                        'lat': coords.get('latitude'),
                        'lon': coords.get('longitude'),
                        'pm25': r.get('value', 0.0),
                        'unit': r.get('unit', 'µg/m³')
                    })
            return pd.DataFrame(records).drop_duplicates(subset=['lat', 'lon'])
    except Exception as e:
        return pd.DataFrame()

def generate_fallback_sensors(lat, lon, radius_km):
    """Generates realistic local grid sensors if OpenAQ has no data in range."""
    np.random.seed(int(abs(lat * 100)))
    num_sensors = np.random.randint(8, 25)
    records = []
    deg_range = radius_km / 111.0

    for i in range(num_sensors):
        r_lat = lat + np.random.uniform(-deg_range, deg_range) * 0.7
        r_lon = lon + np.random.uniform(-deg_range, deg_range) * 0.7
        pm25_val = max(5.0, round(np.random.normal(24.0, 15.0), 1))
        records.append({
            'location': f"Station Simulated-{100+i}",
            'lat': r_lat,
            'lon': r_lon,
            'pm25': pm25_val,
            'unit': 'µg/m³'
        })
    return pd.DataFrame(records)

def get_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def get_risk_color(val):
    if val < 12.0: return 'green'
    if val < 35.4: return 'orange'
    if val < 55.4: return 'red'
    return 'purple'

def handle_search(b):
    with dashboard_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)
        print("Geocoding and fetching live AQI data... Please wait.")

    loc = geocode_address(address_input.value)
    if not loc:
        with metrics_output:
            clear_output(wait=True)
            print("Error: Location not found. Please check spelling or query format.")
        return

    lat, lon, disp_name = loc
    radius_val = radius_km_input.value
    df_live = fetch_openaq_data(lat, lon, radius_val)
    is_fallback = False

    if df_live.empty:
        df_live = generate_fallback_sensors(lat, lon, radius_val)
        is_fallback = True

    with metrics_output:
        clear_output(wait=True)
        print(f"Target Location: {disp_name}")
        print(f"Coordinates: Lat {lat:.4f}, Lon {lon:.4f}")
        print("-" * 50)
        if is_fallback:
            print("[Note] No live OpenAQ stations found. Localized fallback grid simulated.")

        num_sensors = len(df_live)
        area_km2 = np.pi * (radius_val ** 2)
        sensor_density = num_sensors / area_km2

        min_res = float('inf')
        if num_sensors >= 2:
            for i in range(num_sensors):
                for j in range(i + 1, num_sensors):
                    d = get_distance(
                        df_live.iloc[i]['lat'], df_live.iloc[i]['lon'],
                        df_live.iloc[j]['lat'], df_live.iloc[j]['lon']
                    )
                    if d < min_res:
                        min_res = d

        resolution_str = f"{min_res:.2f} km" if min_res != float('inf') else "N/A"
        print(f"Active Sensors in Zone: {num_sensors}")
        print(f"Monitored Area: {area_km2:.1f} km²")
        print(f"Sensor Density: {sensor_density:.4f} sensors/km²")
        print(f"Spatial Resolution (Min Distance): {resolution_str}")

    with dashboard_output:
        clear_output(wait=True)

        # --- Robust Mapbox Integration Check ---
        mapbox_token = None
        try:
            mapbox_token = userdata.get('MAP_API_KEY')
        except Exception:
            pass

        if mapbox_token:
            # Render dark, high-contrast, premium map style utilizing client-provided Mapbox Token
            m = folium.Map(
                location=[lat, lon],
                zoom_start=11,
                tiles='https://api.mapbox.com/styles/v1/mapbox/dark-v10/tiles/{z}/{x}/{y}?access_token=' + mapbox_token,
                attr='Mapbox'
            )
        else:
            # Clean CartoDB fallback
            m = folium.Map(location=[lat, lon], zoom_start=11, tiles='CartoDB positron')

        heat_data = [[row['lat'], row['lon'], row['pm25']] for _, row in df_live.iterrows()]
        HeatMap(heat_data, radius=25, blur=15, min_opacity=0.4).add_to(m)

        for _, row in df_live.iterrows():
            popup_info = f"""
            <div style='font-family: Arial, sans-serif;'>
                <h4>{row['location']}</h4>
                <b>Current PM2.5:</b> {row['pm25']:.1f} µg/m³<br>
                <b>Risk Level:</b> <span style='color:{get_risk_color(row['pm25'])}'><b>{get_risk_color(row['pm25']).upper()}</b></span>
            </div>
            """
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=6,
                color='white' if mapbox_token else 'black',
                weight=1,
                fill=True,
                fill_color=get_risk_color(row['pm25']),
                fill_opacity=0.9,
                popup=folium.Popup(popup_info, max_width=250)
            ).add_to(m)

        folium.Circle(
            location=[lat, lon],
            radius=radius_val * 1000,
            color='cyan' if mapbox_token else 'blue',
            weight=1.5,
            fill=True,
            fill_opacity=0.03,
            popup=f"Target boundary: {radius_val}km"
        ).add_to(m)

        display(m)

search_button.on_click(handle_search)

controls_box = widgets.VBox([
    widgets.HBox([address_input, radius_km_input]),
    search_button
])

display(controls_box)
display(metrics_output)
display(dashboard_output)
handle_search(None)